# YAXCHÉ: Simulacion Espacio-Temporal de Incendios en Guatemala
### Pipeline de Referencia Cientifica y Georreferenciada (WGS84)

> **Marco Conceptual:** En la cosmovision maya, la **Yaxche (la Ceiba Pentandra)** es el Arbol de la Vida que conecta las estrellas con la tierra viva. **YAXCHÉ** acopla datos de observacion de la Tierra (**NASA SRTM, FIRMS, NOAA GFS**) con un automata celular 2D y el modelo fisico de Rothermel para anticipar la propagacion de frentes termicos a escala territorial en Guatemala previo a su aceleracion en el cluster Kabre.

## 1. Importacion de Modulos Cientificos

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import time
import numpy as np
import matplotlib.pyplot as plt

from src.terrain import TerrainGrid
from src.wind import WindField
from src.simulation import FireSimulation, STATE_UNBURNED, STATE_BURNING, STATE_BURNED, run_benchmark
from src.visualization import plot_scientific_fire_report

%matplotlib inline
print("Modulos cientificos de YAXCHE cargados exitosamente.")

## 2. Dominio Territorial Georreferenciado de Guatemala (WGS84)

Generamos la malla territorial oficial delimitada por la frontera soberana de Guatemala:
- **Sistema de Coordenadas:** WGS84 (Latitud 13.73° N a 17.82° N, Longitud -92.25° W a -88.22° W).
- **Escala Metrica:** 1 celda = 100 m x 100 m = 1 hectarea ($10,000 \text{ m}^2$).
- **Relieve Regional:** Tierras bajas de Peten (norte), Cadena Volcanica de la Sierra Madre y Cuchumatanes (sur-centro).

In [ ]:
filas, cols = 350, 350
terreno = TerrainGrid.create_guatemala_domain(rows=filas, cols=cols)

ha_gt = int(np.sum(terreno.mask_guatemala))
print(f"Dominio territorial generado: {ha_gt:,} hectareas activas dentro de Guatemala.")

fig, ax = plt.subplots(1, 2, figsize=(14, 6), facecolor="#F8FAFC")
extent = [terreno.LON_MIN, terreno.LON_MAX, terreno.LAT_MIN, terreno.LAT_MAX]

# Topografia continental
topo_mask = np.copy(terreno.elevation)
topo_mask[~terreno.mask_guatemala] = np.nan
im0 = ax[0].imshow(topo_mask, extent=extent, cmap='gist_earth', origin='upper')
ax[0].set_title("Orografia Continental de Guatemala (m s.n.m.)", fontsize=11, fontweight="bold")
ax[0].set_xlabel("Longitud Oeste (WGS84)")
ax[0].set_ylabel("Latitud Norte (WGS84)")
plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04, label="Metros s.n.m.")

# Densidad de Combustible Forestal
fuel_mask = np.copy(terreno.fuel)
fuel_mask[~terreno.mask_guatemala] = np.nan
im1 = ax[1].imshow(fuel_mask, extent=extent, cmap='YlGn', origin='upper')
ax[1].set_title("Densidad de Combustible Forestal (Biomasa NDVI)", fontsize=11, fontweight="bold")
ax[1].set_xlabel("Longitud Oeste (WGS84)")
ax[1].set_ylabel("Latitud Norte (WGS84)")
plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04, label="Coeficiente [0.0 - 1.0]")
plt.tight_layout()
plt.show()

## 3. Simulacion de Propagacion e Isocronas de Tiempo de Llegada (TOA)

Iniciamos dos focos de ignicion reales georreferenciados:
1. **Reserva de la Biosfera Maya (Peten):** 17.20° N, -90.50° W.
2. **Cadena Volcanica / Sierra Madre:** 14.55° N, -91.10° W.

Configuramos un campo de viento representativo de vientos alisios (14 km/h hacia el Noreste).

In [ ]:
viento = WindField.uniform(shape=(filas, cols), speed_ms=4.0, direction_degrees=45.0)
sim = FireSimulation(terrain=terreno, wind=viento, step_duration_hours=1.0)

# Igniciones georreferenciadas
sim.ignite_coord(lon=-90.50, lat=17.20)  # Peten
sim.ignite_coord(lon=-91.10, lat=14.55)  # Altiplano

horas = 24
print(f"Ejecutando simulacion fisica de {horas} horas...")
t0 = time.perf_counter()
historial = sim.run(steps=horas)
t_total = time.perf_counter() - t0

stats_fin = historial[-1]
print(f"Simulacion completada en {t_total:.4f} s ({t_total/horas:.4f} s/hora simulada).")
print(f"Hectareas consumidas acumuladas: {stats_fin['total_affected_ha']:,} ha ({stats_fin['total_affected_km2']:.2f} km²).")
print(f"Frentes activos en combustion: {stats_fin['active_front_ha']:,} ha.")

# Generar reporte cartografico estandar FARSITE / EFFIS
fig_rep = plot_scientific_fire_report(sim, history=historial)
plt.show()

## 4. Benchmark de Escalabilidad en CPU Secuencial

Evaluacion cuantitativa del tiempo de ejecucion al incrementar la resolucion sobre el dominio oficial de Guatemala. Esta medicion constituye el 'código victima' que se sometera al perfilador de rendimiento y a la paralelizacion en el cluster Kabre.

In [ ]:
resultados = run_benchmark(sizes=[200, 300, 400], steps=5)

tamanios = [r[0] for r in resultados]
tiempos = [r[3] for r in resultados]

plt.figure(figsize=(8, 4), dpi=100, facecolor="white")
plt.plot(tamanios, tiempos, 'o-', color='#DC2626', linewidth=2, label='CPU Monohilo (Python Secuencial)')
plt.title("Cuello de Botella Computacional en CPU Monohilo", fontsize=11, fontweight="bold")
plt.xlabel("Dimension de Malla (N x N)", fontsize=9)
plt.ylabel("Tiempo Total de Calculo (s)", fontsize=9)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Agenda de Aceleracion en el Cluster Kabre (CeNAT)

1. **Bloque Matutino (CPU):**
   - Perfilado con `cProfile` y `line_profiler` para identificar el costo de acceso a memoria en los bucles de vecindad.
   - Paralelizacion con `@numba.jit(parallel=True)` y `prange` sobre los nucleos del nodo Kabre.
   - Medicion de curvas de escalamiento fuerte y debil (Ley de Amdahl).
2. **Bloque Vespertino (GPU):**
   - Migracion de mallas matriciales a la memoria de video con CuPy (`cp.ndarray`).
   - Implementacion del Kernel CUDA 2D Stencil donde cada celda de Guatemala es evaluada concurrentemente por un hilo de GPU.
   - Medicion de latencia de transferencia PCIe (Host-a-Device) vs tiempo neto de computo.